In [ ]:
from pathlib import Path
import pandas as pd

INPUT_DIR = Path("/kaggle/input")

FILENAMES = [
    "qwen_all_report_states.csv",
    "qwen_state_calibration.csv",
    "qwen_verified_evaluation.csv",
]

paths = {}

# Search a few folder levels without scanning the MRI directories
for filename in FILENAMES:
    matches = set()

    for depth in range(1, 5):
        pattern = "/".join(["*"] * depth + [filename])
        matches.update(INPUT_DIR.glob(pattern))

    candidates = sorted(matches)

    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one copy of {filename}, found "
            f"{len(candidates)}: {candidates}. "
            "Check the attached notebook outputs."
        )

    paths[filename] = candidates[0]
    print(filename, "->", candidates[0])

states = pd.read_csv(
    paths[FILENAMES[0]],
    dtype={"StudyInstanceUID": str},
)

# Preserve the literal 'null' calibration state
calibration = pd.read_csv(
    paths[FILENAMES[1]],
    keep_default_na=False,
    dtype={"State": str},
)

evaluation = pd.read_csv(
    paths[FILENAMES[2]],
    dtype={"StudyInstanceUID": str},
)

assert len(states) == 4407, "Unexpected report count."
assert states["StudyInstanceUID"].is_unique, "Duplicate study IDs."
assert len(evaluation) == 57, "Unexpected evaluation count."

errors = states["ParsingError"].fillna("").astype(str).str.strip()
assert errors.eq("").all(), "Some reports have extraction errors."

print("\nFiles loaded:")
print("Report states:", states.shape)
print("Calibration:", calibration.shape)
print("Evaluation:", evaluation.shape)
print("\nInitial checks passed.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

train = pd.read_csv(
    DATA_DIR / "train.csv",
    dtype={"StudyInstanceUID": str},
)

sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

LABELS = [
    column
    for column in sample_submission.columns
    if column != "StudyInstanceUID"
]

# Confirm the extracted states cover exactly the training studies
assert train["StudyInstanceUID"].is_unique
assert set(states["StudyInstanceUID"]) == set(
    train["StudyInstanceUID"]
), "Training IDs and report-state IDs do not match."

# Check every extracted value is 0, 1, or missing
for label in LABELS:
    column = f"state_{label}"
    numeric = pd.to_numeric(states[column], errors="raise")

    assert numeric.dropna().isin([0, 1]).all(), (
        f"Invalid values found for {label}"
    )

    states[column] = numeric

# Compare full-run states against the earlier evaluation
assert evaluation["StudyInstanceUID"].is_unique

comparison = evaluation.merge(
    states[
        ["StudyInstanceUID"]
        + [f"state_{label}" for label in LABELS]
    ],
    on="StudyInstanceUID",
    how="inner",
    validate="one_to_one",
)

assert len(comparison) == 57

agreement_rows = []

for label in LABELS:
    earlier = pd.to_numeric(
        comparison[f"pred_{label}"], errors="raise"
    )
    full_run = comparison[f"state_{label}"]

    matches = (
        earlier.eq(full_run)
        | (earlier.isna() & full_run.isna())
    )

    agreement_rows.append({
        "Label": label,
        "Matching": int(matches.sum()),
        "Different": int((~matches).sum()),
        "Agreement": matches.mean(),
    })

agreement = pd.DataFrame(agreement_rows).set_index("Label")

display(
    agreement.style.format({"Agreement": "{:.1%}"})
)

print("Total differing label states:", agreement["Different"].sum())

# Count available official supervision
official_mask = train[LABELS].notna().all(axis=1)

print("\nFully officially labelled studies:", official_mask.sum())
print(
    "Studies with no official labels:",
    train[LABELS].isna().all(axis=1).sum(),
)

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

# ---------------------------------------------------------
# Combine official labels with the completed report states
# ---------------------------------------------------------

state_columns = [f"state_{label}" for label in LABELS]

training_table = train.merge(
    states[["StudyInstanceUID"] + state_columns],
    on="StudyInstanceUID",
    how="left",
    validate="one_to_one",
)

training_table["HasOfficialLabels"] = (
    training_table[LABELS].notna().all(axis=1)
)

# Group identical reports conservatively to reduce leakage.
# Matching text does not necessarily mean the same patient.
normalized_reports = (
    training_table["Report"]
    .fillna("")
    .astype(str)
    .str.normalize("NFKC")
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Empty reports should not all become one group
training_table["ReportGroup"] = [
    hashlib.sha256(
        (
            report
            if report
            else "missing:" + study_id
        ).encode("utf-8")
    ).hexdigest()
    for report, study_id in zip(
        normalized_reports,
        training_table["StudyInstanceUID"],
    )
]

# ---------------------------------------------------------
# Keep the prompt-development study out of validation
# ---------------------------------------------------------

DEVELOPMENT_ID = (
    "1.2.826.0.1.3680043.8.498."
    "10095687747295410396510538520594649149"
)

development_rows = training_table[
    training_table["StudyInstanceUID"] == DEVELOPMENT_ID
]

assert len(development_rows) == 1

development_group = development_rows.iloc[0]["ReportGroup"]

eligible = (
    training_table["HasOfficialLabels"]
    & training_table["ReportGroup"].ne(development_group)
)

validation_groups = np.array(
    sorted(training_table.loc[eligible, "ReportGroup"].unique())
)

# ---------------------------------------------------------
# Assign groups to three reproducible validation folds
# ---------------------------------------------------------

splitter = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

group_to_fold = {}

for fold, (_, validation_indices) in enumerate(
    splitter.split(validation_groups)
):
    for group in validation_groups[validation_indices]:
        group_to_fold[group] = fold

# -1 means the group is never held out.
# Unlabelled reports matching a validation report inherit its fold.
training_table["Fold"] = (
    training_table["ReportGroup"]
    .map(group_to_fold)
    .fillna(-1)
    .astype(int)
)

# ---------------------------------------------------------
# Audit each split and its label balance
# ---------------------------------------------------------

split_summary = []
label_balance = []

for fold in range(3):
    train_mask = training_table["Fold"].ne(fold)

    validation_mask = (
        training_table["Fold"].eq(fold)
        & training_table["HasOfficialLabels"]
    )

    train_part = training_table[train_mask]
    validation_part = training_table[validation_mask]

    assert set(train_part["ReportGroup"]).isdisjoint(
        set(validation_part["ReportGroup"])
    )

    assert DEVELOPMENT_ID not in set(
        validation_part["StudyInstanceUID"]
    )

    split_summary.append({
        "Fold": fold,
        "Official training": int(
            train_part["HasOfficialLabels"].sum()
        ),
        "Unlabelled training": int(
            (~train_part["HasOfficialLabels"]).sum()
        ),
        "Official validation": len(validation_part),
    })

    for label in LABELS:
        positives = int(validation_part[label].eq(1).sum())
        negatives = int(validation_part[label].eq(0).sum())

        label_balance.append({
            "Fold": fold,
            "Label": label,
            "Positive": positives,
            "Negative": negatives,
            "AUC computable": positives > 0 and negatives > 0,
        })

split_summary = pd.DataFrame(split_summary)
label_balance = pd.DataFrame(label_balance)

display(split_summary)

display(
    label_balance.pivot(
        index="Label",
        columns="Fold",
        values="Positive",
    ).rename_axis(columns="Validation fold: positive counts")
)

problem_rows = label_balance[
    ~label_balance["AUC computable"]
]

if problem_rows.empty:
    print("Every validation fold has positives and negatives for all labels.")
else:
    print("Warning: these fold/label combinations cannot produce ROC-AUC:")
    display(problem_rows)

print("\nNo identical-report overlap between training and validation.")
print("Splits prepared in memory; no existing files were changed.")

In [ ]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Baseline settings — experimental, not validated confidence
# ---------------------------------------------------------

ACTIVE_FOLD = 0
PRIOR_STRENGTH = 8.0
WEAK_SCALE = 0.25

# Exclude the entire held-out report group from training
train_part = training_table[
    training_table["Fold"].ne(ACTIVE_FOLD)
].copy().reset_index(drop=True)

validation_part = training_table[
    training_table["Fold"].eq(ACTIVE_FOLD)
    & training_table["HasOfficialLabels"]
].copy().reset_index(drop=True)

official_mask = train_part["HasOfficialLabels"]
official_training = train_part[official_mask]

assert len(official_training) == 39
assert len(validation_part) == 19

# Safe placeholder targets; zero weight means ignored.
train_targets = pd.DataFrame(
    0.5, index=train_part.index, columns=LABELS
)

train_weights = pd.DataFrame(
    0.0, index=train_part.index, columns=LABELS
)

calibration_rows = []

# ---------------------------------------------------------
# Fit mappings using ONLY official training labels
# ---------------------------------------------------------

for label in LABELS:
    state_column = f"state_{label}"

    base_rate = float(official_training[label].mean())

    for state in [0, 1]:
        matches = official_training[state_column].eq(state)
        count = int(matches.sum())

        # Unobserved states remain unused
        if count == 0:
            continue

        positives = int(
            official_training.loc[matches, label].eq(1).sum()
        )

        # Shrink the small-sample estimate toward training prevalence
        probability = (
            positives + PRIOR_STRENGTH * base_rate
        ) / (count + PRIOR_STRENGTH)

        sample_support = count / (count + PRIOR_STRENGTH)

        # Heuristic measure of separation from the base rate
        max_distance = (
            1.0 - base_rate
            if probability >= base_rate
            else base_rate
        )

        separation = (
            abs(probability - base_rate) / max_distance
            if max_distance > 0
            else 0.0
        )

        weight = WEAK_SCALE * sample_support * separation

        # Apply only to unlabelled training studies
        apply_mask = (
            ~official_mask
            & train_part[state_column].eq(state)
        )

        train_targets.loc[apply_mask, label] = probability
        train_weights.loc[apply_mask, label] = weight

        calibration_rows.append({
            "Label": label,
            "State": state,
            "Official examples": count,
            "Official positives": positives,
            "Soft target": probability,
            "Training weight": weight,
            "Unlabelled studies assigned": int(apply_mask.sum()),
        })

    # Official labels always override report-derived targets
    train_targets.loc[official_mask, label] = (
        train_part.loc[official_mask, label].astype(float)
    )
    train_weights.loc[official_mask, label] = 1.0

fold_calibration = pd.DataFrame(calibration_rows)

# Validation uses official labels, never pseudo-labels
validation_targets = validation_part[LABELS].astype(float).copy()

# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert np.isfinite(train_targets.to_numpy()).all()
assert np.isfinite(train_weights.to_numpy()).all()

assert (
    (train_targets.to_numpy() >= 0)
    & (train_targets.to_numpy() <= 1)
).all()

assert (
    (train_weights.to_numpy() >= 0)
    & (train_weights.to_numpy() <= 1)
).all()

assert np.array_equal(
    train_targets.loc[official_mask].to_numpy(),
    train_part.loc[official_mask, LABELS].to_numpy(dtype=float),
)

assert set(train_part["ReportGroup"]).isdisjoint(
    set(validation_part["ReportGroup"])
)

# ---------------------------------------------------------
# Display the training supervision available per label
# ---------------------------------------------------------

weak_weights = train_weights.loc[~official_mask]

supervision_summary = pd.DataFrame({
    "Official training": int(official_mask.sum()),
    "Weak labels used": weak_weights.gt(0).sum(),
    "Weak labels ignored": weak_weights.eq(0).sum(),
    "Total weak weight": weak_weights.sum(),
    "Validation positives": validation_targets.eq(1).sum(),
})

display(
    supervision_summary.style.format({
        "Total weak weight": "{:.2f}"
    })
)

print("\nFold:", ACTIVE_FOLD)
print("Training studies:", len(train_part))
print("Validation studies:", len(validation_part))
print("Targets shape:", train_targets.shape)
print("Weights shape:", train_weights.shape)
print("\nFold-specific targets prepared; existing files unchanged.")

In [ ]:
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

PLANES = ["Sagittal", "Coronal", "Axial"]
N_SLICES = 16
IMAGE_SIZE = 224

train_series = pd.read_csv(
    DATA_DIR / "train_series.csv",
    dtype={
        "StudyInstanceUID": str,
        "SeriesInstanceUID": str,
    },
)


def select_study_series(study_id):
    """Select one series per plane using a simple baseline rule."""
    rows = train_series[
        train_series["StudyInstanceUID"].eq(study_id)
    ].copy()

    # Prefer fluid-sensitive, then fat-suppressed acquisitions.
    rows = rows.sort_values(
        ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"],
        ascending=[False, False, True],
    )

    selected = []

    for plane in PLANES:
        candidates = rows[rows["Anatomical_Plane"].eq(plane)]

        if candidates.empty:
            raise ValueError(f"{study_id}: missing {plane} series")

        selected.append(candidates.iloc[0])

    return pd.DataFrame(selected).reset_index(drop=True)


def ordered_slice_paths(series_folder):
    """Sort slices using their position along the slice normal."""
    paths = sorted(series_folder.glob("*.dcm"))

    if not paths:
        raise FileNotFoundError(f"No DICOM files: {series_folder}")

    headers = [
        pydicom.dcmread(path, stop_before_pixels=True)
        for path in paths
    ]

    has_geometry = all(
        hasattr(ds, "ImageOrientationPatient")
        and hasattr(ds, "ImagePositionPatient")
        for ds in headers
    )

    if has_geometry:
        orientations = np.array([
            list(ds.ImageOrientationPatient) for ds in headers
        ], dtype=float)

        if not np.allclose(
            orientations, orientations[0], atol=1e-3
        ):
            raise ValueError(
                f"Mixed slice orientations in {series_folder}"
            )

        normal = np.cross(
            orientations[0, :3], orientations[0, 3:]
        )

        if np.linalg.norm(normal) < 1e-6:
            raise ValueError("Invalid slice orientation metadata.")

        positions = np.array([
            list(ds.ImagePositionPatient) for ds in headers
        ], dtype=float)

        order = np.argsort(positions @ normal, kind="stable")

    elif all(hasattr(ds, "InstanceNumber") for ds in headers):
        print("Warning: using InstanceNumber fallback:", series_folder.name)
        order = np.argsort(
            [float(ds.InstanceNumber) for ds in headers],
            kind="stable",
        )
    else:
        raise ValueError(
            f"Insufficient slice-order metadata: {series_folder}"
        )

    return [paths[i] for i in order]


def load_series(series_folder):
    paths = ordered_slice_paths(series_folder)

    # Uniformly sample across the stack; repeat slices if necessary.
    indices = np.rint(
        np.linspace(0, len(paths) - 1, N_SLICES)
    ).astype(int)

    images = []
    interpretations = []

    for index in indices:
        ds = pydicom.dcmread(paths[index])
        image = ds.pixel_array.astype(np.float32)

        if image.ndim != 2:
            raise ValueError(f"Expected a 2D slice: {paths[index]}")

        image = (
            image * float(getattr(ds, "RescaleSlope", 1))
            + float(getattr(ds, "RescaleIntercept", 0))
        )

        images.append(image)
        interpretations.append(
            str(getattr(ds, "PhotometricInterpretation", ""))
        )

    if len(set(interpretations)) != 1:
        raise ValueError("Mixed photometric interpretations.")

    if interpretations[0] not in ("MONOCHROME1", "MONOCHROME2"):
        raise ValueError("Expected grayscale MRI slices.")

    volume = np.stack(images)

    if not np.isfinite(volume).all():
        raise ValueError("Non-finite pixel values.")

    # Normalize jointly across the sampled series, not slice by slice.
    low, high = np.percentile(volume, [1, 99])

    if high <= low:
        raise ValueError("Series has no usable intensity variation.")

    volume = np.clip((volume - low) / (high - low), 0, 1)

    if interpretations[0] == "MONOCHROME1":
        volume = 1.0 - volume

    tensor = torch.from_numpy(volume).float().unsqueeze(1)

    # Resize while preserving the pixel-grid aspect ratio, then pad.
    height, width = tensor.shape[-2:]
    scale = IMAGE_SIZE / max(height, width)
    new_h = max(1, round(height * scale))
    new_w = max(1, round(width * scale))

    tensor = F.interpolate(
        tensor,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False,
    )

    pad_h = IMAGE_SIZE - new_h
    pad_w = IMAGE_SIZE - new_w

    tensor = F.pad(
        tensor,
        (
            pad_w // 2, pad_w - pad_w // 2,
            pad_h // 2, pad_h - pad_h // 2,
        ),
    )

    return tensor, len(paths)


# Test on an officially labelled training study
example_row = train_part[
    train_part["HasOfficialLabels"]
].iloc[0]

example_id = example_row["StudyInstanceUID"]
selected_series = select_study_series(example_id)

volumes = []

for _, row in selected_series.iterrows():
    folder = (
        DATA_DIR / "train_series" / example_id
        / row["SeriesInstanceUID"]
    )

    volume, original_count = load_series(folder)
    volumes.append(volume)

    print(
        row["Anatomical_Plane"],
        "| original slices:", original_count,
        "| tensor:", tuple(volume.shape),
    )

study_tensor = torch.stack(volumes)

assert study_tensor.shape == (3, 16, 1, 224, 224)
assert torch.isfinite(study_tensor).all()

print("\nStudy tensor:", tuple(study_tensor.shape))
print("Order: planes, slices, channels, height, width")

fig, axes = plt.subplots(3, 4, figsize=(10, 8))

for plane_index, plane in enumerate(PLANES):
    for column, slice_index in enumerate([3, 6, 9, 12]):
        axes[plane_index, column].imshow(
            study_tensor[plane_index, slice_index, 0].numpy(),
            cmap="gray",
            vmin=0,
            vmax=1,
        )
        axes[plane_index, column].set_title(
            f"{plane} — slice {slice_index}"
        )
        axes[plane_index, column].axis("off")

plt.tight_layout()
plt.show()

plt.close(fig)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import resnet18, ResNet18_Weights

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)


# ---------------------------------------------------------
# Load one complete study using the previous cell's functions
# ---------------------------------------------------------

def load_study_tensor(study_id):
    selected = select_study_series(study_id)
    volumes = []

    for _, row in selected.iterrows():
        folder = (
            DATA_DIR / "train_series" / study_id
            / row["SeriesInstanceUID"]
        )
        volume, _ = load_series(folder)
        volumes.append(volume)

    return torch.stack(volumes)


# ---------------------------------------------------------
# Frozen slice encoder + trainable study classifier
# ---------------------------------------------------------

class KneeBaseline(nn.Module):
    def __init__(self, number_of_labels):
        super().__init__()

        self.encoder = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )
        feature_size = self.encoder.fc.in_features
        self.encoder.fc = nn.Identity()

        for parameter in self.encoder.parameters():
            parameter.requires_grad = False

        self.head = nn.Sequential(
            nn.Linear(3 * feature_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, number_of_labels),
        )

        # ImageNet channel normalization.
        # Keep the loader's existing resize/padding for this baseline.
        self.register_buffer(
            "image_mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1),
        )
        self.register_buffer(
            "image_std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1),
        )

    def forward(self, images):
        # Shape: batch, planes, slices, channels, height, width
        batch, planes, slices, channels, height, width = images.shape
        assert planes == 3 and channels == 1

        flat_images = images.reshape(-1, 1, height, width)

        # Frozen encoder: also freeze batch-normalization statistics.
        self.encoder.eval()
        features = []

        with torch.no_grad():
            # Small chunks limit encoder memory use.
            for chunk in flat_images.split(16):
                rgb = chunk.repeat(1, 3, 1, 1)
                rgb = (rgb - self.image_mean) / self.image_std
                features.append(self.encoder(rgb))

        features = torch.cat(features, dim=0)
        features = features.reshape(batch, planes, slices, -1)

        # Average slices within each plane, preserving plane identity.
        study_features = features.mean(dim=2).flatten(start_dim=1)

        return self.head(study_features)


mri_model = KneeBaseline(len(LABELS)).to(device)


# ---------------------------------------------------------
# Select one official and one usable weakly labelled study
# ---------------------------------------------------------

official_indices = train_part.index[
    train_part["HasOfficialLabels"]
]

weak_indices = train_part.index[
    ~train_part["HasOfficialLabels"]
    & train_weights.sum(axis=1).gt(0)
]

batch_indices = [
    int(official_indices[0]),
    int(weak_indices[0]),
]

batch_images = []

for index in batch_indices:
    study_id = train_part.loc[index, "StudyInstanceUID"]
    print("Loading:", study_id)
    batch_images.append(load_study_tensor(study_id))

images = torch.stack(batch_images).to(device)

targets = torch.tensor(
    train_targets.loc[batch_indices, LABELS].to_numpy(
        dtype=np.float32
    ),
    device=device,
)

weights = torch.tensor(
    train_weights.loc[batch_indices, LABELS].to_numpy(
        dtype=np.float32
    ),
    device=device,
)

is_official = torch.tensor(
    train_part.loc[
        batch_indices, "HasOfficialLabels"
    ].to_numpy(dtype=bool),
    device=device,
)


# ---------------------------------------------------------
# One optimization step with separately normalized losses
# ---------------------------------------------------------

optimizer = torch.optim.AdamW(
    mri_model.head.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

# Controls the relative influence of the normalized weak loss.
# This is a starting hyperparameter, not a validated optimum.
WEAK_LOSS_MULTIPLIER = 0.5

mri_model.train()
optimizer.zero_grad(set_to_none=True)

logits = mri_model(images)

element_loss = F.binary_cross_entropy_with_logits(
    logits, targets, reduction="none"
)

official_loss = element_loss[is_official].mean()

weak_element_loss = element_loss[~is_official]
weak_element_weights = weights[~is_official]

weak_loss = (
    (weak_element_loss * weak_element_weights).sum()
    / weak_element_weights.sum().clamp_min(1e-8)
)

loss = official_loss + WEAK_LOSS_MULTIPLIER * weak_loss

assert torch.isfinite(loss), "Non-finite loss."

loss.backward()

head_gradients = [
    parameter.grad
    for parameter in mri_model.head.parameters()
    if parameter.requires_grad
]

assert all(
    gradient is not None and torch.isfinite(gradient).all()
    for gradient in head_gradients
), "Missing or non-finite classifier gradients."

gradient_norm = torch.nn.utils.clip_grad_norm_(
    mri_model.head.parameters(), max_norm=1.0
)

optimizer.step()

print("\nInput shape:", tuple(images.shape))
print("Output shape:", tuple(logits.shape))
print(f"Official loss: {official_loss.item():.4f}")
print(f"Weak loss: {weak_loss.item():.4f}")
print(f"Combined loss: {loss.item():.4f}")
print(f"Gradient norm: {float(gradient_norm):.4f}")
print("\nOne training step completed successfully.")

In [ ]:
import gc
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm


# ---------------------------------------------------------
# Configuration
# Change the version if you change preprocessing or the encoder.
# ---------------------------------------------------------

FEATURE_PATH = Path(
    "/kaggle/working/resnet18_mri_features_v1.npz"
)

SAVE_EVERY = 25
FEATURE_DIM = 3 * 512

feature_config = {
    "version": "resnet18_mri_features_v1",
    "encoder": "ResNet18_Weights.IMAGENET1K_V1",
    "planes": list(PLANES),
    "slices": N_SLICES,
    "image_size": IMAGE_SIZE,
    "pooling": "mean_slices_then_concatenate_planes",
    "preprocessing": "existing_load_series_v1",
    "precision": "float32",
}

config_text = json.dumps(feature_config, sort_keys=True)

assert torch.cuda.is_available(), "Enable GPU before this run."

mri_model = mri_model.to("cuda")
mri_model.eval()

study_ids = training_table["StudyInstanceUID"].astype(str).to_numpy()
assert len(set(study_ids)) == len(study_ids)

number_of_studies = len(study_ids)

features = np.zeros(
    (number_of_studies, FEATURE_DIM),
    dtype=np.float32,
)

completed = np.zeros(number_of_studies, dtype=bool)
errors = [""] * number_of_studies


# ---------------------------------------------------------
# Restore an existing checkpoint, if available
# ---------------------------------------------------------

if FEATURE_PATH.exists():
    with np.load(FEATURE_PATH, allow_pickle=False) as saved:
        assert str(saved["config"].item()) == config_text, (
            "Checkpoint configuration differs. Use a new output filename."
        )
        assert np.array_equal(saved["study_ids"], study_ids), (
            "Checkpoint study IDs or ordering differ."
        )

        features = saved["features"].copy()
        completed = saved["completed"].copy()
        errors = saved["errors"].tolist()

    assert features.shape == (number_of_studies, FEATURE_DIM)
    assert completed.shape == (number_of_studies,)
    assert np.isfinite(features[completed]).all()

    print("Resuming completed studies:", int(completed.sum()))

else:
    print("Starting a new feature extraction.")

print("Studies remaining:", int((~completed).sum()))
print("Output:", FEATURE_PATH)


# ---------------------------------------------------------
# Atomic checkpoint: replace the previous file only after
# the new checkpoint has been fully written.
# ---------------------------------------------------------

def save_feature_checkpoint():
    temporary_path = FEATURE_PATH.with_suffix(".tmp")

    with open(temporary_path, "wb") as handle:
        np.savez_compressed(
            handle,
            study_ids=study_ids.astype(str),
            features=features,
            completed=completed,
            errors=np.asarray(errors, dtype=str),
            config=np.asarray(config_text),
        )

    os.replace(temporary_path, FEATURE_PATH)


# ---------------------------------------------------------
# Extract exactly the features used by KneeBaseline
# ---------------------------------------------------------

@torch.no_grad()
def encode_study(study_id):
    volume = load_study_tensor(study_id)

    assert tuple(volume.shape) == (
        3, N_SLICES, 1, IMAGE_SIZE, IMAGE_SIZE
    )

    slices = volume.reshape(
        -1, 1, IMAGE_SIZE, IMAGE_SIZE
    )

    encoded_chunks = []

    for chunk in slices.split(16):
        chunk = chunk.to("cuda", dtype=torch.float32)
        rgb = chunk.repeat(1, 3, 1, 1)

        rgb = (
            rgb - mri_model.image_mean
        ) / mri_model.image_std

        encoded_chunks.append(
            mri_model.encoder(rgb).cpu()
        )

    encoded = torch.cat(encoded_chunks, dim=0)
    encoded = encoded.reshape(3, N_SLICES, 512)

    # Preserve sagittal/coronal/axial identity.
    pooled = encoded.mean(dim=1).reshape(-1).numpy()

    assert pooled.shape == (FEATURE_DIM,)
    assert np.isfinite(pooled).all()

    return pooled


# ---------------------------------------------------------
# Process pending studies
# Failed studies remain incomplete and are retried on rerun.
# ---------------------------------------------------------

pending_indices = np.flatnonzero(~completed)

started = time.perf_counter()
attempted = 0
consecutive_failures = 0

progress = tqdm(
    pending_indices,
    desc="Extracting MRI features",
    unit="study",
)

try:
    for index in progress:
        study_id = study_ids[index]

        try:
            features[index] = encode_study(study_id)
            completed[index] = True
            errors[index] = ""
            consecutive_failures = 0

        except Exception as error:
            errors[index] = f"{type(error).__name__}: {error}"
            consecutive_failures += 1

            tqdm.write(
                f"Failed {study_id}: {errors[index][:350]}"
            )

            gc.collect()
            torch.cuda.empty_cache()

            # Stop a potentially systemic failure instead of
            # silently generating thousands of incomplete rows.
            if consecutive_failures >= 5:
                raise RuntimeError(
                    "Five consecutive studies failed. "
                    "Progress will be saved; inspect the errors."
                ) from error

        attempted += 1

        if attempted % SAVE_EVERY == 0:
            save_feature_checkpoint()

        if attempted % 10 == 0:
            elapsed = time.perf_counter() - started
            seconds_per_study = elapsed / attempted

            remaining_this_run = (
                len(pending_indices) - attempted
            )

            progress.set_postfix({
                "complete": int(completed.sum()),
                "sec/study": f"{seconds_per_study:.1f}",
                "hours_left": (
                    f"{remaining_this_run * seconds_per_study / 3600:.2f}"
                ),
            })

finally:
    # Saves on normal completion or a catchable interruption.
    # A forced session shutdown can still lose draft-session files.
    save_feature_checkpoint()
    progress.close()


# ---------------------------------------------------------
# Final integrity report
# ---------------------------------------------------------

print("\nFeature extraction finished.")
print("Completed studies:", int(completed.sum()))
print("Incomplete studies:", int((~completed).sum()))
print("Feature matrix:", features.shape)
print("Saved:", FEATURE_PATH)

if completed.all():
    assert np.isfinite(features).all()
    print("\nAll MRI features are ready for classifier training.")
else:
    print("\nIncomplete studies must be resolved before training.")

    for index in np.flatnonzero(~completed)[:10]:
        print(study_ids[index], errors[index])

In [ ]:
from pathlib import Path
import pandas as pd
import pydicom

DATA_DIR = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

FAILED_IDS = [
    "1.2.826.0.1.3680043.8.498.34685905030370793639196564723935583035",
    "1.2.826.0.1.3680043.8.498.37833587429731221455928642963031995680",
]

metadata = pd.read_csv(
    DATA_DIR / "train_series.csv",
    dtype={
        "StudyInstanceUID": str,
        "SeriesInstanceUID": str,
    },
)

official = pd.read_csv(
    DATA_DIR / "train.csv",
    dtype={"StudyInstanceUID": str},
)

labels = [
    column for column in official.columns
    if column not in ["StudyInstanceUID", "Report"]
]

summary = []
failure_details = []

for study_id in FAILED_IDS:
    official_row = official[
        official["StudyInstanceUID"].eq(study_id)
    ].iloc[0]

    print("\nStudy:", study_id)
    print(
        "Has official labels:",
        bool(official_row[labels].notna().all()),
    )

    # Reproduce the original series-selection rule
    candidates = metadata[
        metadata["StudyInstanceUID"].eq(study_id)
    ].sort_values(
        ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"],
        ascending=[False, False, True],
    )

    for plane in ["Sagittal", "Coronal", "Axial"]:
        series = candidates[
            candidates["Anatomical_Plane"].eq(plane)
        ].iloc[0]

        folder = (
            DATA_DIR / "train_series" / study_id
            / series["SeriesInstanceUID"]
        )

        paths = sorted(folder.glob("*.dcm"))
        successful = 0
        failed = 0

        # Inspect every slice in the selected series
        for path in paths:
            ds = None

            try:
                ds = pydicom.dcmread(path)
                pixels = ds.pixel_array
                successful += 1

            except Exception as error:
                failed += 1

                detail = {
                    "StudyInstanceUID": study_id,
                    "Plane": plane,
                    "File": str(path),
                    "Error": str(error),
                }

                if ds is not None:
                    for field in [
                        "Rows",
                        "Columns",
                        "BitsAllocated",
                        "BitsStored",
                        "HighBit",
                        "PixelRepresentation",
                        "SamplesPerPixel",
                        "NumberOfFrames",
                        "PhotometricInterpretation",
                    ]:
                        detail[field] = str(
                            getattr(ds, field, "not present")
                        )

                    detail["TransferSyntaxUID"] = str(
                        ds.file_meta.get("TransferSyntaxUID", "")
                    )
                    detail["PixelDataBytes"] = len(
                        ds.get("PixelData", b"")
                    )

                failure_details.append(detail)

        summary.append({
            "StudyInstanceUID": study_id,
            "Plane": plane,
            "Files": len(paths),
            "Decoded": successful,
            "Failed": failed,
        })

diagnostic_summary = pd.DataFrame(summary)
diagnostic_failures = pd.DataFrame(failure_details)

display(diagnostic_summary)

# Show one failing slice per affected series
if not diagnostic_failures.empty:
    examples = diagnostic_failures.drop_duplicates(
        ["StudyInstanceUID", "Plane"]
    )

    for _, example in examples.iterrows():
        print("\nFailing-slice metadata:")
        print(example.to_string())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

FILENAME = "resnet18_mri_features_v1.npz"
feature_path = Path("/kaggle/working") / FILENAME

# If using a new session, attach the completed notebook output.
if not feature_path.exists():
    candidates = set()

    for depth in range(1, 5):
        pattern = "/".join(["*"] * depth + [FILENAME])
        candidates.update(Path("/kaggle/input").glob(pattern))

    if len(candidates) != 1:
        raise RuntimeError(
            "Attach the completed feature-extraction notebook output. "
            f"Found {len(candidates)} matching feature files."
        )

    feature_path = next(iter(candidates))

# Load the checkpoint without changing it
with np.load(feature_path, allow_pickle=False) as checkpoint:
    feature_ids = checkpoint["study_ids"].astype(str)
    feature_matrix = checkpoint["features"].copy()
    feature_complete = checkpoint["completed"].astype(bool)
    feature_config_text = str(checkpoint["config"].item())

assert len(feature_ids) == len(set(feature_ids))
assert feature_matrix.shape == (len(feature_ids), 1536)
assert feature_complete.shape == (len(feature_ids),)
assert np.isfinite(feature_matrix[feature_complete]).all()

# Ensure we loaded features for exactly this training dataset
assert set(feature_ids) == set(
    training_table["StudyInstanceUID"].astype(str)
)

id_to_feature_row = {
    study_id: index
    for index, study_id in enumerate(feature_ids)
}

excluded_ids = set(feature_ids[~feature_complete])

excluded_studies = training_table[
    training_table["StudyInstanceUID"].isin(excluded_ids)
]

assert not excluded_studies["HasOfficialLabels"].any(), (
    "An officially labelled study is incomplete; investigate before training."
)

print("Excluded unlabelled studies:", len(excluded_ids))

# Preserve the exact correspondence between rows, labels, and weights
assert train_part.index.equals(train_targets.index)
assert train_part.index.equals(train_weights.index)
assert validation_part.index.equals(validation_targets.index)

keep_training = ~train_part["StudyInstanceUID"].isin(excluded_ids)

fit_rows = train_part.loc[keep_training].copy()
fit_targets = train_targets.loc[keep_training, LABELS].copy()
fit_weights = train_weights.loc[keep_training, LABELS].copy()

assert not validation_part["StudyInstanceUID"].isin(excluded_ids).any()

train_feature_indices = np.array([
    id_to_feature_row[study_id]
    for study_id in fit_rows["StudyInstanceUID"]
])

validation_feature_indices = np.array([
    id_to_feature_row[study_id]
    for study_id in validation_part["StudyInstanceUID"]
])

assert feature_complete[train_feature_indices].all()
assert feature_complete[validation_feature_indices].all()

X_train = feature_matrix[train_feature_indices].astype(np.float32)
X_validation = feature_matrix[
    validation_feature_indices
].astype(np.float32)

y_train = fit_targets.to_numpy(dtype=np.float32)
w_train = fit_weights.to_numpy(dtype=np.float32)
y_validation = validation_targets[LABELS].to_numpy(dtype=np.float32)

is_official_train = fit_rows[
    "HasOfficialLabels"
].to_numpy(dtype=bool)

assert set(fit_rows["ReportGroup"]).isdisjoint(
    set(validation_part["ReportGroup"])
)

assert X_train.shape[0] == y_train.shape[0] == w_train.shape[0]

print("\nFold:", ACTIVE_FOLD)
print("Training features:", X_train.shape)
print("Official training studies:", int(is_official_train.sum()))
print("Unlabelled training studies:", int((~is_official_train).sum()))
print("Validation features:", X_validation.shape)
print("Training targets:", y_train.shape)
print("Training weights:", w_train.shape)
print("\nFeature alignment passed. Original checkpoint unchanged.")